<a href="https://colab.research.google.com/github/BrynX5/econ-project/blob/main/labor_market_forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecasting U.S. Unemployment from Labor Market Indicators

**Question:** Can last quarter's labor market and macro indicators forecast next quarter's unemployment rate better than assuming it stays the same?

**Approach**
1. Pull 9 public series from FRED (no API key needed) and aggregate them to quarters.
2. Use each indicator **one quarter earlier** to predict the **change** in unemployment, so this is a true forecast.
3. Evaluate with a **walk-forward test**: retrain every quarter on only past data, then predict the next quarter.
4. Compare a random forest against two baselines: *no change* (next quarter = this quarter) and a linear model.

All errors are in **percentage points** of the unemployment rate.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

Path("figures").mkdir(exist_ok=True)
Path("results").mkdir(exist_ok=True)
SEED = 42

## 1. Data

Each series is read from FRED's public CSV download, so no API key is needed. Series observed more often than quarterly are averaged within each quarter.

In [ ]:
SERIES = {
    "unemployment":   "UNRATE",           # unemployment rate, %, monthly
    "job_openings":   "JTSJOL",           # JOLTS job openings, thousands, monthly (starts Dec 2000)
    "cpi":            "CPIAUCSL",         # CPI, index, monthly
    "gdp_growth":     "A191RL1Q225SBEA",  # real GDP growth, annualized %, quarterly
    "lfpr":           "CIVPART",          # labor force participation rate, %, monthly
    "avg_wages":      "CES0500000003",    # average hourly earnings, $, monthly
    "yield_spread":   "T10Y2Y",           # 10-year minus 2-year Treasury, %, daily
    "initial_claims": "ICSA",             # initial jobless claims, weekly
    "pce":            "PCE",              # personal consumption expenditures, $bn, monthly
}

def fred(series_id: str) -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    raw = pd.read_csv(url)
    s = pd.to_numeric(raw.iloc[:, 1], errors="coerce")   # FRED marks missing days with "."
    s.index = pd.to_datetime(raw.iloc[:, 0])
    return s.dropna()

raw = {name: fred(code) for name, code in SERIES.items()}
q = pd.DataFrame({name: s.resample("QS").mean() for name, s in raw.items()}).dropna()
q = q[q.index < pd.Timestamp.today().to_period("Q").start_time]   # drop the incomplete current quarter
print(f"{len(q)} quarters, {q.index.min().to_period('Q')} to {q.index.max().to_period('Q')}")
q.tail()

## 2. Features: last quarter's indicators, this quarter's change

Price, wage and spending levels trend upward forever, so they enter as year-over-year growth rates. Every feature is lagged one quarter. The target is the **change** in unemployment, which keeps the random forest from having to extrapolate to levels it has never seen.

In [ ]:
f = pd.DataFrame(index=q.index)
f["unemployment"]       = q["unemployment"]
f["d_unemployment"]     = q["unemployment"].diff()
f["job_openings_yoy"]   = q["job_openings"].pct_change(4) * 100
f["inflation_yoy"]      = q["cpi"].pct_change(4) * 100
f["gdp_growth"]         = q["gdp_growth"]
f["lfpr"]               = q["lfpr"]
f["wage_growth_yoy"]    = q["avg_wages"].pct_change(4) * 100
f["yield_spread"]       = q["yield_spread"]
f["claims_yoy"]         = q["initial_claims"].pct_change(4) * 100
f["pce_growth_yoy"]     = q["pce"].pct_change(4) * 100

FEATURES = [c for c in f.columns]                 # all of them, as of LAST quarter
X = f[FEATURES].shift(1).add_suffix("_lag1")
data = X.assign(target=f["d_unemployment"], u_prev=f["unemployment"].shift(1),
                u_now=f["unemployment"]).dropna()
print(f"{len(data)} usable quarters after lags, {data.index.min().to_period('Q')} to {data.index.max().to_period('Q')}")
data.head()

## 3. Walk-forward test

The last 20% of quarters are the test period. For each test quarter, every model is retrained on all earlier quarters only, then forecasts that one quarter. This is how the model would have performed in real time.

In [ ]:
feat_cols = [c for c in data.columns if c.endswith("_lag1")]
n_test = max(8, int(round(len(data) * 0.2)))
test_idx = data.index[-n_test:]

models = {
    "random_forest": lambda: RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=SEED),
    "linear_ridge":  lambda: make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
}

rows = []
for t in test_idx:
    train = data[data.index < t]
    row = {"quarter": t, "actual": data.at[t, "u_now"], "no_change": data.at[t, "u_prev"]}
    for name, make in models.items():
        m = make().fit(train[feat_cols], train["target"])
        row[name] = data.at[t, "u_prev"] + m.predict(data.loc[[t], feat_cols])[0]
    rows.append(row)
pred = pd.DataFrame(rows).set_index("quarter")

mae = {name: float((pred[name] - pred["actual"]).abs().mean()) for name in ["no_change", *models]}
results = pd.DataFrame({"MAE (pct points)": mae}).sort_values("MAE (pct points)")
results["vs no-change baseline"] = (results["MAE (pct points)"] / mae["no_change"] - 1).map("{:+.0%}".format)
print(f"Test period: {test_idx.min().to_period('Q')} to {test_idx.max().to_period('Q')} ({n_test} quarters), one quarter ahead")
results

## 4. Which indicators matter?

The random forest is refit on all quarters to rank features by importance. Importance shows what the model relies on, not cause and effect.

In [ ]:
rf_full = models["random_forest"]().fit(data[feat_cols], data["target"])
imp = (pd.Series(rf_full.feature_importances_, index=[c.replace("_lag1", "") for c in feat_cols])
         .sort_values())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
ax.plot(pred.index, pred["actual"], color="black", lw=2, label="Actual")
ax.plot(pred.index, pred["random_forest"], color="#3b6ea8", lw=1.6, label=f"Random forest (MAE {mae['random_forest']:.2f})")
ax.plot(pred.index, pred["no_change"], color="#999", lw=1.2, ls="--", label=f"No change (MAE {mae['no_change']:.2f})")
ax.set_title("One-quarter-ahead unemployment forecasts, walk-forward test", loc="left")
ax.set_ylabel("Unemployment rate (%)")
ax.legend(frameon=False)

ax = axes[1]
ax.barh(imp.index, imp.values, color="#3b6ea8")
ax.set_title("Random forest feature importance", loc="left")
ax.set_xlabel("Share of importance (last quarter's values)")
for a in axes:
    a.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("figures/forecast_and_importance.png", dpi=200)
plt.show()

In [ ]:
summary = {
    "quarters_used": int(len(data)),
    "sample": f"{data.index.min().to_period('Q')} to {data.index.max().to_period('Q')}",
    "test_period": f"{test_idx.min().to_period('Q')} to {test_idx.max().to_period('Q')}",
    "test_quarters": int(n_test),
    "mae_pct_points": {k: round(v, 3) for k, v in mae.items()},
    "rf_vs_no_change": round(mae["random_forest"] / mae["no_change"] - 1, 3),
    "top_features": imp.sort_values(ascending=False).head(3).round(3).to_dict(),
}
Path("results/metrics.json").write_text(json.dumps(summary, indent=2))
pred.round(3).to_csv("results/walk_forward_predictions.csv")
print(json.dumps(summary, indent=2))

## Limits

- **Small sample:** about 100 quarters, of which roughly 20 are tested. Treat differences of a few hundredths of a point as noise.
- **The COVID shock:** 2020 is in the training data. The unemployment spike that year is far outside normal variation and can distort what the models learn.
- **Revised data:** FRED serves today's revised figures, not what forecasters saw at the time, so real-time accuracy would be somewhat worse.
- **Importance is not causality:** the feature importance chart shows what the model relies on, not why unemployment moves.